# Qwen on Colab - first test (exercise version)

**You type the code, the notebook explains why.** Every code cell is empty except for a list of hints
(*what to write*). Try it yourself first; if you get stuck, open **"Stuck? Show the reference code"** under the cell.

The long prompt cells in Step 7 are already filled in - they are data, not code to write.

The full, ready-to-run version is `qwen_digest_test.ipynb` if you just want to run everything.

## Step 1 - Do we really have a GPU?

**Why:** A language model is billions of numbers, and writing one word means multiplying all of them together.
A CPU does that one piece at a time; a GPU does thousands of pieces at the same moment. Without the GPU this test
would take hours instead of seconds. The GPU has its own memory (**VRAM**), and that number decides which models fit.

In [ ]:
# YOUR TURN - Step 1
# 1. import os, shutil, subprocess, time, json (we use all of them later)
# 2. run the command-line tool `nvidia-smi` with subprocess.run(...)
#    ask it for the GPU name and memory.total, as csv without a header: --query-gpu=name,memory.total --format=csv,noheader
#    use capture_output=True, text=True
# 3. print what it answered
# 4. assert the command succeeded (returncode == 0) so a missing GPU stops us right here
# 5. turn the memory text ("15360 MiB") into GB and keep it in gpu_gb  -> Step 3 needs it



<details><summary>Stuck? Show the reference code</summary>

```python
import os, shutil, subprocess, time, json

# ask the GPU what it is and how much memory it has
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True)
print(result.stdout)

# no GPU -> stop here
assert result.returncode == 0, "No GPU. Runtime > Change runtime type > pick a GPU"

# the memory comes as text like "15360 MiB"; turn it into GB
memory_text = result.stdout.split(",")[1]
gpu_gb = int(memory_text.split()[0]) / 1024
print("GPU memory:", round(gpu_gb), "GB")
```

</details>

## Step 2 - Connect Google Drive

**Why:** Colab gives you a *rented* computer. When the session ends (or you are idle for a while) the whole machine
is wiped, including anything you downloaded. Drive is your permanent shelf: the model goes there once, and every
future session just takes it from the shelf instead of downloading it again.

A window will ask you to allow access - that is expected.

In [ ]:
# YOUR TURN - Step 2
# 1. from google.colab import drive  ->  drive.mount("/content/drive")
# 2. DRIVE_DIR = our own folder: /content/drive/MyDrive/AQ-Worker
# 3. os.makedirs(..., exist_ok=True) for DRIVE_DIR/models and DRIVE_DIR/runs
# 4. shutil.disk_usage("/content/drive") -> print free and total space in GB



<details><summary>Stuck? Show the reference code</summary>

```python
from google.colab import drive
drive.mount("/content/drive")

# our own folder, so we never mix with your other files
DRIVE_DIR = "/content/drive/MyDrive/AQ-Worker"
os.makedirs(DRIVE_DIR + "/models", exist_ok=True)   # the model file lives here
os.makedirs(DRIVE_DIR + "/runs", exist_ok=True)     # saved prompt -> answer records live here

disk = shutil.disk_usage("/content/drive")
print("Drive free:", round(disk.free / 1e9, 1), "GB of", round(disk.total / 1e9, 1), "GB")
```

</details>

## Step 3 - Pick the model (and what "14B" and "Q4" mean)

* **14B** = 14 billion *parameters*. A parameter is one learned number; more of them usually means a smarter model,
  but also a bigger file and slower writing.
* **GGUF / Q4_K_M** = the model with each number *rounded* to about 4 bits instead of 16. Think of saving a photo as a
  smaller JPEG: the file is about 4x smaller and quality drops only a little. That is why a 14B model fits in about **9 GB**
  instead of about 28 GB.

**Rule of thumb: the model file must be smaller than the GPU memory, with a few GB to spare** for the model's working notes
(the context). With Colab Pro you may get a bigger GPU than the free T4, so the cell below suggests the strongest model
that fits the GPU you actually got:

| GPU memory | Suggested model | File size (approx.) | Note |
|---|---|---|---|
| 38 GB or more (A100) | `qwen3-32b` | ~20 GB | Strongest of these; all 32B numbers are used for every word |
| 22 to 37 GB (L4) | `qwen3-30b-a3b` | ~19 GB | "Mixture of experts": 30B in total but only ~3B used per word, so it writes fast |
| under 22 GB (T4) | `qwen3-14b` | ~9 GB | Free-tier choice |

**Drive space:** free Drive is 15 GB shared with Gmail and Photos. The 20 GB models need a bigger Google storage plan.
Colab Pro does not include Drive storage, that is a separate Google One plan. Step 4 checks before it downloads.

In [ ]:
# YOUR TURN - Step 3
# 1. a dict MODELS: name -> (repo, file name) for qwen3-32b, qwen3-30b-a3b, qwen3-14b, qwen3-8b
#    (Hugging Face names look like repo "Qwen/Qwen3-14B-GGUF" and file "Qwen3-14B-Q4_K_M.gguf")
# 2. model_name from gpu_gb, with if / elif / else:  >= 38 -> 32b,  >= 22 -> 30b-a3b,  otherwise 14b
# 3. repo, filename = MODELS[model_name]  and print what we picked



<details><summary>Stuck? Show the reference code</summary>

```python
# name -> (Hugging Face repo, file name)
MODELS = {
    "qwen3-32b":     ("Qwen/Qwen3-32B-GGUF",     "Qwen3-32B-Q4_K_M.gguf"),      # ~20 GB, needs an A100
    "qwen3-30b-a3b": ("Qwen/Qwen3-30B-A3B-GGUF", "Qwen3-30B-A3B-Q4_K_M.gguf"),  # ~19 GB, fast, needs ~24 GB GPU
    "qwen3-14b":     ("Qwen/Qwen3-14B-GGUF",     "Qwen3-14B-Q4_K_M.gguf"),      # ~9 GB, fits a T4
    "qwen3-14b-q6":  ("Qwen/Qwen3-14B-GGUF",     "Qwen3-14B-Q6_K.gguf"),        # ~12 GB, same model rounded less, a T4 can still hold it
    "qwen3-8b":     ("Qwen/Qwen3-8B-GGUF",      "Qwen3-8B-Q4_K_M.gguf"),       # ~5 GB, smallest
}

# pick the biggest model that fits the GPU we got
if gpu_gb >= 38:
    model_name = "qwen3-32b"
elif gpu_gb >= 22:
    model_name = "qwen3-30b-a3b"
else:
    model_name = "qwen3-14b"

# model_name = "qwen3-14b-q6"  # <- uncomment to try the less-rounded 14B (about 12 GB, tight on a 16 GB GPU)
# model_name = "qwen3-8b"      # <- uncomment to force a smaller model and compare

repo, filename = MODELS[model_name]
print("using", model_name, "->", repo, "/", filename)
```

</details>

## Step 4 - Download the model into Drive (one time)

**Why we check the size first:** a half-downloaded 20 GB file wastes your time and your Drive space. We ask Hugging Face
(the site that hosts the model) how big the file is, compare it with your free Drive space, and only then download.
If the file is already on Drive from an earlier run, this step does nothing.

In [ ]:
# YOUR TURN - Step 4
# 1. %pip install -q -U huggingface_hub, then import HfApi and hf_hub_download
# 2. HfApi().model_info(repo, files_metadata=True)  ->  build a dict {file name: size in bytes}
# 3. if our file name is not in that dict: raise an error that lists the .gguf files that DO exist
# 4. if the file is already on Drive with exactly the right size: skip.
#    else: check Drive free space is at least size * 1.05 (raise an error if not), then hf_hub_download(repo_id=..., filename=..., local_dir=DRIVE_DIR/models)
# 5. at the end assert the size on Drive equals the expected size (a different size = broken download)



<details><summary>Stuck? Show the reference code</summary>

```python
%pip install -q -U huggingface_hub
from huggingface_hub import HfApi, hf_hub_download

# how big is the file on Hugging Face?
files = HfApi().model_info(repo, files_metadata=True).siblings
sizes = {f.rfilename: f.size for f in files}

if filename not in sizes:
    print("File not found. These .gguf files exist in that repo:")
    for name in sizes:
        if name.endswith(".gguf"):
            print("  ", name)
    raise ValueError("fix the file name in Step 3")

model_size = sizes[filename]
drive_file = DRIVE_DIR + "/models/" + filename
print("model file:", round(model_size / 1e9, 1), "GB")

if os.path.exists(drive_file) and os.path.getsize(drive_file) == model_size:
    print("already on Drive, nothing to download")
else:
    free = shutil.disk_usage("/content/drive").free
    if free < model_size * 1.05:
        raise ValueError("Not enough Drive space. Pick a smaller model in Step 3.")
    start = time.time()
    hf_hub_download(repo_id=repo, filename=filename, local_dir=DRIVE_DIR + "/models")
    print("downloaded in", round(time.time() - start), "seconds")

# a file with the wrong size means a broken download
assert os.path.getsize(drive_file) == model_size, "broken download - delete the file from Drive and run again"
print("OK:", drive_file)
```

</details>

## Step 5 - Copy to the Colab disk, then load into the GPU

**Two moves, two reasons:**

1. *Drive -> Colab's own disk.* Reading a 20 GB file straight from Drive is slow and flaky, a local copy is fast.
   This copy is repeated each session; it is not a download.
2. *Disk -> GPU memory ("loading the model").* The model is just a big file of numbers. Loading means putting those
   numbers into the GPU's memory so the GPU can calculate with them. `n_gpu_layers=-1` means "put every layer on the GPU".

**`n_ctx` (context size)** is how much text the model can look at in one go: the prompt AND its own answer together.
A bigger value costs more GPU memory (the model keeps notes about every word it has seen). Our prompt is about 2k tokens
and we allow about 1.5k tokens of answer, so 8192 is comfortable.

A **token** is a chunk of a word: roughly 3/4 of an English word, or about 4 characters.

In [ ]:
# YOUR TURN - Step 5 (copy)
# 1. os.makedirs("/content/models"), then local_file = /content/models/<filename>
# 2. if it is not there yet (or its size differs from the Drive file): shutil.copyfile(drive_file, local_file) and time it



<details><summary>Stuck? Show the reference code</summary>

```python
os.makedirs("/content/models", exist_ok=True)
local_file = "/content/models/" + filename

if os.path.exists(local_file) and os.path.getsize(local_file) == model_size:
    print("already copied")
else:
    start = time.time()
    shutil.copyfile(drive_file, local_file)
    print("copied in", round(time.time() - start), "seconds")
print("local file:", local_file)
```

</details>

### Install the program that runs the model (`llama-cpp-python`)

This is the engine that reads the GGUF file and does the math on the GPU. We try a ready-made GPU build first
(takes about a minute). If it complains, use the fallback cell right below (compiles from source, about 15 minutes).

In [ ]:
# YOUR TURN - install the engine
# 1. %pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
# 2. import llama_cpp, print llama_cpp.__version__ and llama_cpp.llama_supports_gpu_offload()  (must say True)



<details><summary>Stuck? Show the reference code</summary>

```python
%pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

import llama_cpp
print("version:", llama_cpp.__version__)
print("can use GPU:", llama_cpp.llama_supports_gpu_offload())
```

</details>

In [ ]:
# ONLY run this cell if the one above failed or printed "can use GPU: False".
# It compiles the engine with CUDA on this machine. Slow once, then it works.
# %env CMAKE_ARGS=-DGGML_CUDA=on
# %pip install -q --no-cache-dir --force-reinstall llama-cpp-python

In [ ]:
# YOUR TURN - load the model into the GPU
# 1. from llama_cpp import Llama
# 2. llm = Llama(model_path=local_file, n_gpu_layers=-1, n_ctx=8192, verbose=False)   # time it and print the seconds



<details><summary>Stuck? Show the reference code</summary>

```python
from llama_cpp import Llama

start = time.time()
llm = Llama(
    model_path=local_file,
    n_gpu_layers=-1,   # -1 = put every layer on the GPU
    n_ctx=8192,        # room for prompt + answer, counted in tokens
    verbose=False,
)
print("loaded in", round(time.time() - start), "seconds")
```

</details>

## Step 6 - Say hello (the simplest possible test)

**Why:** before we use a long, complicated prompt, make sure the basics work. If this prints a sensible reply,
the GPU, the file and the engine are all fine.

Note `/no_think`: Qwen3 can "think out loud" before answering, which costs time and tokens. For note-making we want the
answer directly, so we switch the thinking off with that tag.

In [ ]:
# YOUR TURN - Step 6
# 1. write ask(system, user, max_tokens=1500, temperature=0.2) that returns (text, tokens_written, seconds)
# 2. inside: llm.create_chat_completion(messages=[system message, user message + "\n/no_think"], max_tokens=..., temperature=...)
# 3. the text is reply["choices"][0]["message"]["content"], the tokens are reply["usage"]["completion_tokens"]
# 4. remove leftover <think> and </think> tags from the text
# 5. call it with "Say hello in one short sentence." and print the text and the time



<details><summary>Stuck? Show the reference code</summary>

```python
def ask(system, user, max_tokens=1500, temperature=0.2):
    """Send one prompt to the model. Returns the answer text, tokens written and seconds taken."""
    start = time.time()
    reply = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user + "\n/no_think"},   # /no_think = skip the thinking, answer directly
        ],
        max_tokens=max_tokens,     # never write more than this many tokens
        temperature=temperature,   # 0 = always the safest word, higher = more varied. Low is right for notes.
    )
    seconds = time.time() - start

    text = reply["choices"][0]["message"]["content"] or ""
    text = text.replace("<think>", "").replace("</think>", "").strip()
    tokens = reply["usage"]["completion_tokens"]
    return text, tokens, seconds


text, tokens, seconds = ask("You are a helpful assistant.", "Say hello in one short sentence.", max_tokens=60)
print(text)
print(tokens, "tokens in", round(seconds, 1), "s")
```

</details>

## Step 7 - The real input: one slice of a real lecture + the real prompt

**Where it comes from:** this is the exact kind of input our pipeline sends to Groq today. The video is a 4-hour
system-design lecture; the pipeline cuts the transcript into slices of about 5,000 characters, and each slice goes through
the **digest** prompt ("extract only the teachable facts as JSON"). Below is slice 3 of 44 (starting at 09:38).

Two parts make up the prompt:

* **System message** = the *job description* ("you are stage one of a note pipeline, keep only knowledge, do not invent facts").
* **User message** = the *task of this call* (the transcript slice and the JSON shape we want back).

Keep this exact text. Later we will change the prompt and compare answers, and we can only compare against something that stayed fixed.

In [ ]:
SYSTEM_PROMPT = r'''You are the first stage of a lecture-note pipeline. You read ONE slice of a transcript and extract ONLY its teachable knowledge as compact, faithful facts that a second stage will turn into notebook pages.

Drop intros, sponsors, subscribe asks, filler, jokes, tangents and anecdotes with no teaching value. Do not invent or add outside facts; fix only obvious auto-caption spelling of technical terms. Write facts in your own short words (no long verbatim quotes), in English. Keep numbers, names, commands, versions and code exactly as stated. Use the [mm:ss] markers for `start`.
Output ONE JSON object only.'''

In [ ]:
USER_TEMPLATE = r'''Lecture: title: System Design Crash Course in 4 Hours; channel: Mohit Chhabra; length: 3:45:12
This is slice 3 of 44. (Slices overlap slightly at the edges: do not worry about a cut-off first/last sentence.)

Extract the teachable knowledge as 1-6 topics, in order. Total output must stay under about 181 words, so be selective and dense: prefer facts, definitions, steps, numbers and code over narration. If the slice contains NO teachable content (pure intro/outro/sponsor), return {"topics":[]}.

JSON shape (omit empty/null optional fields):
{"topics":[{"heading":"short topic name","start":"mm:ss","facts":["one self-contained fact per string"],"terms":[{"term":"x","definition":"y"}],"steps":["ordered step, only if a process was explained"],"example":"one illustrative example or null","code":[{"language":"py","code":"only code actually shown"}],"table":{"headers":["a","b"],"rows":[["x","y"]]} or null,"formula":"only if stated or null","diagram_hint":"if a process/architecture/comparison exists: type (flow|stack|compare|cycle|tree|timeline|mindmap|sequence) + its parts, else null"}]}

TRANSCRIPT SLICE:
"""
{{CHUNK}}
"""'''

In [ ]:
TRANSCRIPT_SLICE = r'''[09:38] Like create, read, update, delete. So these major operations are performed on the database because it contains the data. You can create more data. You can read the data. Whatever existing data is present, you can read that. Then you can update some of the data or you can also delete some of the data. Okay. So this is a very basic overview like how things work, right? And maybe most of you already know this flow. Okay. So this part basically request and response. [10:11] This is basically API, right? So I can write it here. Uh you already may know this. There is client. There is server uh there is database then there is API so in API when client is asking something from the server it's called request and when server gives back the output it's called the response okay very basic overview right so let's move forward with it because we want to get more details in it so our major flow in our question is related to this side you can say uh like we simply said from [10:46] client a request will go to the server and server will give back the response, right? But what actually happens in between like how does this all happen? Uh so before going to that part, uh generally like if you want to know the request and response in what format it is, so most of the times it is in JSON uh format. So JSON is used in most of the APIs you would see that's very famous. There are other options as well to send like the request response format but JSON is mostly used. So just to [11:19] showcase you the request response it would be something like this. So here you can see it's a get uh API method and then this is basically the URL and we are using http 1.1 here. Don't worry about all this. I will cover all this information in detail in this video. Then this is the host uh api.example.com. You can imagine it something like www.google.com or youtube.com Facebook.com everything all such things are host. So here the host is api.acample.com. [11:49] Uh then there is authorization content type accept user agent. User agent will basically mean uh from where you are hitting this API. So for example I ran this from postman. So it says postman and here you can see it says JSON. So JSON as I mentioned you it's like a very famous um way in which you send the request in the response. Okay. Um and rest of the stuff don't worry uh we will discuss about each of these things. Um but basically is like host and URL will [12:21] give you like where are you trying to hit the API. HTTP is a protocol. Get is a method. There are other methods as well like post, put, patch, delete uh which can be used for API. And then authorization it's basically to authorize the end user like who is making the API call to authenticate that user we use the authorization which is part of the headers. Um we'll talk about it later don't worry. So this was an example of what client will send to the server that is the request. Now let me [12:52] also show you an example of the response that is what server will send back to the client. Okay. So this is the example of the response. Okay. So first thing is it tells about the protocol same as like HTTP 1.1. Uh then 200 is the status code which basically tells you it was a successful API call that whatever request was sent it was successfully returned as a response. So whatever request was made successful response has been sent from the server side. Okay. [13:24] Again there is content type which is JSON. Uh cache control is there which talks about no cache or cache. You can ignore it for now. Then basically you can see this is the response that is received. It provides a lot of information in it and basically this format is in JSON. Okay. So this is the like just an example of how a request and response looks like. Okay. So let me remove all this and then we can go further in detail about all the components that will be required in the API. So what we are going to talk about is first is the network layer. Okay. So [14:00] network layer will contain the components like pers DNS. Then there is TCP protocol. Then there is TLS protocol that you should be aware about. I will talk about each of these in detail but this is something that we are going to cover about and you should know about all of these fundamentals. So after that we are going to talk about the firewall and after that there will be gateway and then load balancer. So we are going to talk about all of these things u in this video uh in very detail like how each of [14:33] these work and how the complete flow will look like. Okay. So we will start with basically uh DNS. Now to understand this let's see an example. Okay. So here is my channel on YouTube Mohit Chabra Tech. But if you see this URL out there you will see it says https uh colon/www.youtube.com youtube.com/adtheredmoitabra tech right so let me just copy this and paste it here so that we can discuss about it okay so this is the complete'''

In [ ]:
# YOUR TURN - Step 7 (glue)
# 1. USER_PROMPT = USER_TEMPLATE with the text "{{CHUNK}}" replaced by TRANSCRIPT_SLICE
# 2. print the slice length, the whole prompt length in characters, and tokens (about characters // 4)
# 3. print the first 600 characters of the slice so you can see what the model will read



<details><summary>Stuck? Show the reference code</summary>

```python
# put the slice into the template: this is the final text the model will read
USER_PROMPT = USER_TEMPLATE.replace("{{CHUNK}}", TRANSCRIPT_SLICE)

total_chars = len(SYSTEM_PROMPT) + len(USER_PROMPT)
print("transcript slice:", len(TRANSCRIPT_SLICE), "characters")
print("whole prompt:", total_chars, "characters = about", total_chars // 4, "tokens")
print()
print(TRANSCRIPT_SLICE[:600], "...")
```

</details>

## Step 8 - Generate and read the answer

**What to look for with your own eyes** (we will turn each into a number later):

* Is it **valid JSON** in the shape we asked for?
* Are the facts **really in the slice**, or did the model add things from its own memory? (faithfulness)
* Did it **drop the filler** (intros, jokes) and **keep the real teaching** (coverage)?
* How long did it take, and how many **tokens per second**? (speed)

The slice is spoken English with auto-caption mistakes, so also check whether it fixed obvious technical spellings.

In [ ]:
# YOUR TURN - Step 8
# 1. answer, tokens, seconds = ask(SYSTEM_PROMPT, USER_PROMPT, max_tokens=1500, temperature=0.2)
# 2. print the answer, then tokens, seconds and tokens per second



<details><summary>Stuck? Show the reference code</summary>

```python
answer, tokens, seconds = ask(SYSTEM_PROMPT, USER_PROMPT)

print(answer)
print("-" * 60)
print(tokens, "tokens in", round(seconds, 1), "s =", round(tokens / seconds, 1), "tokens per second")
```

</details>

## Step 9 - Quick check: did it return JSON at all?

**Why only this tiny check now:** our pipeline needs to *parse* the answer. If the text is not valid JSON, the whole
slice is wasted and the pipeline would have to retry. This single yes/no is the first metric.

In [ ]:
# YOUR TURN - Step 9
# 1. write try_parse(text): find the first {...} block with a regex (re.S so "." also matches new lines),
#    json.loads it, and return None if anything fails
# 2. parsed = try_parse(answer); print valid JSON yes/no, and list each topic: start, heading, number of facts



<details><summary>Stuck? Show the reference code</summary>

```python
import re

def try_parse(text):
    # the model may add words around the JSON, so take the part from the first { to the last }
    match = re.search(r"\{.*\}", text, re.S)
    if match is None:
        return None
    try:
        return json.loads(match.group(0))
    except ValueError:
        return None


parsed = try_parse(answer)
print("valid JSON:", parsed is not None)

if parsed:
    for topic in parsed.get("topics", []):
        print("-", topic.get("start"), topic.get("heading"), "|", len(topic.get("facts", [])), "facts")
```

</details>

## Step 10 - Save the pair to Drive (this is what the metrics will use later)

**Why:** a metric needs the **input and the output together**, plus *which model and settings* made it. If we only
looked at the screen, the evidence would vanish when the session ends. We append one line per run to a file on Drive
(JSON Lines: one JSON object per line, easy to add to and easy to load later).

**About "loss" and metrics - the intuition:**
*Loss* is a number used while **training** a model: "how wrong was the guess?" We are not training Qwen, so we will not
use training loss. We will use **evaluation metrics** instead: simple scores of the answer compared with the prompt, such as
valid-JSON rate, how many of the slice's key terms appear in the notes (coverage), how many of the notes' claims are
supported by the slice (faithfulness), and speed. They answer "is this model good enough for our notes?"

In [ ]:
# YOUR TURN - Step 10
# 1. build a dict `record` with: when, model, temperature, system_prompt, user_prompt, answer, tokens, seconds, valid_json
# 2. append it as ONE line of JSON to DRIVE_DIR/runs/digest_tests.jsonl
#    (open with "a", write json.dumps(record, ensure_ascii=False) + "\n")



<details><summary>Stuck? Show the reference code</summary>

```python
record = {
    "when": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model": model_name,
    "temperature": 0.2,
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt": USER_PROMPT,
    "answer": answer,
    "tokens": tokens,
    "seconds": round(seconds, 1),
    "valid_json": parsed is not None,
}

# one line per run, so we can keep adding to the file and load it later
with open(DRIVE_DIR + "/runs/digest_tests.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(record, ensure_ascii=False) + "\n")
print("saved")
```

</details>

## Step 11 - Try your own text (optional)

Paste any transcript piece into `MY_TEXT`, run the cell, and read what comes out. Change `temperature` in `ask()` to feel what it does:
0.0 gives the same answer every time; 0.8 gives more varied (and less reliable) answers.

In [ ]:
# YOUR TURN - Step 11
# 1. MY_TEXT = any transcript piece you paste
# 2. run ask() with SYSTEM_PROMPT and USER_TEMPLATE where "{{CHUNK}}" is replaced by MY_TEXT; print the answer and the time



<details><summary>Stuck? Show the reference code</summary>

```python
MY_TEXT = "paste a piece of transcript here"

prompt = USER_TEMPLATE.replace("{{CHUNK}}", MY_TEXT)
text, tokens, seconds = ask(SYSTEM_PROMPT, prompt)
print(text)
print(tokens, "tokens,", round(seconds, 1), "s")
```

</details>

## Step 12 - When you are done

*Runtime > Disconnect and delete runtime* releases the GPU for other people (and keeps your compute units longer).
The model and the saved answers stay on Drive in `AQ-Worker`.

**What to bring back to Claude:** the answer text from Step 8, the speed line, and the "valid JSON" result.